In [2]:
from google.colab import drive
drive.mount("/content/drive")
import pickle, numpy as np, pandas as pd

def macro_f1(y, p, K):
    cm = np.bincount(y * K + p, minlength=K * K).reshape(K, K)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    d = 2 * tp + fp + fn
    return np.mean((2 * tp / np.maximum(d, 1))[d > 0])

def boot_ci(preds, y, B=2000, seed=0):
    # IC 95% del macro-F1 medio entre semillas, remuestreando las ventanas de prueba
    K = int(max(y.max(), max(p.max() for p in preds)) + 1)
    rng = np.random.default_rng(seed); n = len(y); vals = []
    for _ in range(B):
        i = rng.integers(0, n, n)
        vals.append(np.mean([macro_f1(y[i], p[i], K) for p in preds]))
    point = np.mean([macro_f1(y, p, K) for p in preds])
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return 100 * point, 100 * lo, 100 * hi

base = "/content/drive/MyDrive/CAN_IDS_FINAL_EVIDENCE"
jobs = [
    ("REVISION", "R5_chronological_y_win", "R7_ablation_y_win", "embedding(ID) + dt", "OTIDS"),
    ("REVISION", "R5_chronological_y_win", "R6_tuned_y_win", "HistGradientBoosting (tuned)", "OTIDS"),
    ("REVISION", "R5_chronological_y_win", "R5_chronological_y_win", "CNN-LSTM V3-C", "OTIDS"),
    ("REVISION", "R5_chronological_y_win", "R5_chronological_y_win", "Random Forest", "OTIDS"),
    ("CARHACKING", "CH_chronological_y_win", "CH_chronological_y_win", "CNN-LSTM V3-C", "Car-Hacking"),
    ("CARHACKING", "CH_chronological_y_win", "CH_chronological_y_win", "Random Forest", "Car-Hacking"),
    ("CARHACKING", "CH_chronological_y_file", "CH_chronological_y_file", "CNN-LSTM V3-C", "Car-Hacking (file-level)"),
    ("CARHACKING", "CH_chronological_y_file", "CH_chronological_y_file", "Random Forest", "Car-Hacking (file-level)"),
]
cache, rows = {}, []
for folder, ykey, exp, model, ds in jobs:
    if folder not in cache:
        with open(f"{base}/{folder}/revision_predictions.pkl", "rb") as fh:
            cache[folder] = pickle.load(fh)
    P = cache[folder]
    y = np.asarray(P[(ykey, "y_true", 0)], dtype=np.int64)
    preds = [np.asarray(v, dtype=np.int64) for k, v in P.items() if k[0] == exp and k[1] == model]
    if not preds:
        print("Sin predicciones:", ds, model); continue
    pt, lo, hi = boot_ci(preds, y)
    rows.append({"dataset": ds, "model": model, "seeds": len(preds),
                 "macro-F1 (%)": pt, "CI95 low": lo, "CI95 high": hi})
    print(f"{ds:26s} {model:30s} seeds={len(preds)}  {pt:.2f}  [{lo:.2f}, {hi:.2f}]")
ci = pd.DataFrame(rows)
ci.to_csv(f"{base}/FINAL_bootstrap_CI.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OTIDS                      embedding(ID) + dt             seeds=5  99.68  [99.57, 99.78]
OTIDS                      HistGradientBoosting (tuned)   seeds=5  99.66  [99.57, 99.74]
OTIDS                      CNN-LSTM V3-C                  seeds=5  98.89  [98.70, 99.07]
OTIDS                      Random Forest                  seeds=5  74.60  [74.57, 74.63]
Car-Hacking                CNN-LSTM V3-C                  seeds=3  99.91  [99.89, 99.94]
Car-Hacking                Random Forest                  seeds=3  99.82  [99.78, 99.86]
Car-Hacking (file-level)   CNN-LSTM V3-C                  seeds=3  99.38  [99.31, 99.44]
Car-Hacking (file-level)   Random Forest                  seeds=3  99.49  [99.43, 99.54]
